# Module 6 Assignment: Give Your Persona Agent More Tools

In Module 5 you built a Persona Agent with one tool. This assignment makes the same jump Lab 5
made on top of Lab 4: your agent goes from one tool to three, and `chat()` has to learn to route
between them instead of assuming.

You'll reuse your own Module 5 code at almost every step, plus the routing pattern from Lab 5.
This assignment is about **extending** what you already built, not writing it from scratch.

**What you'll build:**
1. Your persona, notes, and first tool, carried over from Module 5
2. A second tool, backed by a free, no-key external API
3. A third tool of your own design
4. JSON descriptions for all three tools
5. An updated system prompt that tells the model when to use each tool
6. A `call_tool(name, arguments)` dispatcher, exactly like Lab 5's
7. A full agent loop that routes across all three tools

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu`, same as every lab and assignment so
far.

If you can find where Lab 5 made this same change to Lab 4, you can make it here too.

## 0. Install the packages (run once)

Same as Module 5, plus one new package: `requests`, for calling your new tool's API.

In [1]:
!uv pip install --upgrade openai pypdf gradio python-dotenv requests

Resolved 63 packages in 680ms
 Downloaded openai
 Downloaded gradio
Prepared 57 packages in 8.65s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 57 packages in 9.05s
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anyio==4.15.1
 + audioop-lts==0.2.2
 + brotli==1.2.0
 + certifi==2026.7.22
 + charset-normalizer==3.5.2
 + click==8.5.0
 + fastapi==0.142.2
 + filelock==4.0.12
 + fsspec==2026.9.0
 + gradio==6.29.1
 + gradio-client==2.7.2
 + groovy==0.1.2
 + h11==0.16.0
 + hf-gradio==0.4.1
 + hf-xet==1.6.0
 + httpcore==1.0.9
 + httpcore2==2.13.1
 + httpx==0.28.1
 + httpx2==2.13.1
 + huggingface-hub==2.1.1
 + idna==3.20
 + jinja2==3.1.6
 + jiter==0.17.0
 + markdown-it-py==4.2.0
 + markupsafe==3.0.4
 + mdurl==0.1.2
 + numpy==2.5.3
 + openai==3.24.0
 + opentelemetry-api==1.45.0
 + orjson==3.12.0
 + pandas=

## 1. Imports

**TODO:** Copy the imports from your Module 5 assignment, plus `requests` for calling an external
API. Add any other standard library module your second or third tool needs (for example `html`,
if the API you pick returns HTML-escaped text the way Open Trivia Database does).

In [2]:
# TODO: import load_dotenv, os, sys, json, PdfReader (from pypdf), gr (gradio), AzureOpenAI,
# and requests
from dotenv import load_dotenv
import os
import sys
import json
import html

import requests
from openai import AzureOpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr


c:\Users\estev\Assignment 6\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Connect to Azure OpenAI via APIM

Identical to every previous lab and assignment. Copy this cell directly from Module 5.

In [4]:
load_dotenv(override=True)

azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_api_key = os.getenv("AZURE_OPENAI_API_KEY")

if not all([azure_endpoint, azure_api_key]):
    sys.exit("Missing settings. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY in your .env file.")

print(f"Azure OpenAI key exists and begins {azure_api_key[:8]}")

# The deployment we're using for this whole assignment - fixed, do not change it
DEPLOYMENT = "gpt-5.4-ptu"

azure_client = AzureOpenAI(
    azure_endpoint=azure_endpoint,
    api_key=azure_api_key,
    api_version="2024-10-21",
)

Azure OpenAI key exists and begins 5b21e970


## 3. Recreate your persona, notes, and first tool from Module 5

Bring forward everything you already built: your `persona`, your `notes/notes.pdf` loading code,
your one tool function from Module 5, and that tool's JSON description. Nothing here needs to
change, this is your starting point.

**TODO:** Paste in, from your Module 5 assignment:
- Your `persona` variable
- Your `PdfReader("notes/notes.pdf")` loading code that builds `notes`
- Your Module 5 tool function (rename it if you like, but keep track of its name)
- That tool's JSON schema dictionary

**Hint:** copy your `notes/notes.pdf` file into this assignment's folder too, in a `notes`
subfolder, exactly like Lab 5 did with Lab 4's notes file.

In [9]:
persona = (
    "your name is Steve Johnson, you talk with profissionalism and always ethical, you often use fancy words to describe your actions and explain your reason"
)

reader = PdfReader("notes/notes.pdf")
notes = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text

# TODO: paste your Module 5 tool function here, for example:
# def my_tool(value):
#     ...
def my_tool(value):
    print(f"Tool called to log a missed question: {value}")

    with open("missed_questions.txt", "a", encoding="utf-8") as f:
        f.write(value + "\n")

    return "the missed questions is for review"

print(persona)
print(notes[:300], "...")

your name is Steve Johnson, you talk with profissionalism and always ethical, you often use fancy words to describe your actions and explain your reason
types
 
of
 
video
 
games
 
genre:
 
 
-
 
Action
 
-
 
Adventure
 
-
 
RPG
 
-
 
Strategy
 
-
 
Simulation
 
-
 
Sports
 
-
 
Racing
 
-
 
Horror
 
-
 
Puzzle
 
-
 
Fighting
 
-
 
Platformer
 
-
 
Shooter
 
-
 
Survival
 
-
 
Stealth
 
-
 
Sandbox
 
-
 
Rhythm
 
 
 
  ...


In [10]:
# TODO: paste your Module 5 tool's JSON schema dictionary here, for example:
# my_tool_json = {
#     "name": "my_tool",
#     "description": "...",
#     "parameters": { ... }
# }
my_tool_json = {
    "name": "my_tool",
    "description": "This tool is used to record any quiz questions that were responded wrong or outside of the list scope, so it can be reviwed later",
    "parameters": {
        "type": "object",
        "properties": {
            "value": {
                "type": "string",
                "description": "the questions in the rank that were outside scope or wrong"
            }
        },
        "required": ["value"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": my_tool_json}]

## 4. New tool: something backed by a free API

Design a **second** tool for your Persona Agent, this one calling a free, no-key external API,
the same way Lab 5's `fetch_bonus_question_tool` calls the Open Trivia Database.

**Ideas, pick whichever fits your topic, or find your own:**
- A trivia-friendly agent could call the Open Trivia Database for a bonus question
- A weather-aware topic could call [Open-Meteo](https://open-meteo.com), a free weather API that
  needs no key
- A language-learning agent could call the [Free Dictionary API](https://dictionaryapi.dev) to
  define a word, though it is sometimes slow or briefly unavailable, so handle that the same way
  the next tip describes
- Any other free, no-key API relevant to your topic works too, just check it does not require
  signup or a paid key

**TODO:** Write your tool function below. It should make one `requests.get(...)` call inside a
`try`/`except` that catches `requests.exceptions.RequestException`, so a slow or unavailable API
returns a short, friendly string instead of crashing your notebook, exactly like Lab 5's tools
do.

In [13]:
def api_tool(value):
    # TODO: implement your tool. Make a requests.get(...) call to a free, no-key API inside a
    # try/except that catches requests.exceptions.RequestException, handle a failed or empty
    # response gracefully, and return a short string either way.
    print(f"Tool called to fetch a bonus trivia question {value}")
    params = {"amount": 1, "type": "multiple", "category": 15}
    try:
        response = requests.get("https://opentdb.com/api.php", params=params, timeout=20)
        data = response.json()
    except (requests.exceptions.RequestException, ValueError) as e:
        print(f"fetch_bonus_question_tool network error: {e}")
        return "The trivia API is temporarily unavailable - just skip the bonus question and continue the quiz."

    if data.get("response_code") != 0 or not data.get("results"):
        return "Could not fetch a bonus question right now - just skip it and continue the quiz."

    q = data["results"][0]
    question = html.unescape(q["question"])
    answer = html.unescape(q["correct_answer"])
    return f"Bonus question: {question} (correct answer: {answer})"

In [14]:
api_tool("test value")  # quick manual test of your API-backed tool

Tool called to fetch a bonus trivia question test value


'Bonus question: Which of the following was a map that was in Team Fortress 2 at launch? (correct answer: Gravel Pit)'

## 5. New tool: one more tool of your own design

Design a **third** tool. It can be another local tool in the spirit of Module 5 (a running score
tracker, a bookmarking tool for topics that need review, or an idea of your own), or a second
external API call if you would rather. Either is fine, as long as it does something the first two
tools do not already cover.

**TODO:** Write your third tool function below.

In [20]:
def third_tool(value):
    # TODO: implement your third tool. Do something small and visible, and return a short string.
    try:
        score = int(value)

        with open("score.txt", "a", encoding="utf-8") as f:
            f.write(f"{score}\n")

        print(f"Tool called to record score: {score}")
        return f"Score of {score} was successfully recorded."

    except ValueError:
        return "Please provide a valid numerical score."

In [21]:
third_tool("test value")  # quick manual test of your third tool

'Please provide a valid numerical score.'

## 6. Describe all three tools to the model with JSON

Same idea as Module 5 and Lab 5, just three schemas now. Remember an argument can be optional (an
empty `"required"` list), exactly like Lab 5's `fetch_bonus_question_tool`, if that fits your
tool.

In [22]:
# TODO: define api_tool_json and third_tool_json, the same shape as your Module 5 tool's
# JSON schema, then build the combined tools list:

# api_tool_json = { "name": "api_tool", "description": "...", "parameters": { ... } }
# third_tool_json = { "name": "third_tool", "description": "...", "parameters": { ... } }

# tools = [
#     {"type": "function", "function": my_tool_json},
#     {"type": "function", "function": api_tool_json},
#     {"type": "function", "function": third_tool_json},
# ]
my_tool_json = {
    "name": "my_tool",
    "description": "Use this tool to record a quiz question the student got wrong or did not know, so they can review it later",
    "parameters": {
        "type": "object",
        "properties": {
            "value": {
                "type": "string",
                "description": "The exact quiz question the student missed"
            }
        },
        "required": ["value"],
        "additionalProperties": False
    }
}

api_tool_json = {
    "name": "api_tool",
    "description": "Use this tool to fetch a bonus video game trivia question from an external trivia database",
    "parameters": {
        "type": "object",
        "properties": {
            "value": {
                "type": "string",
                "description": "The student's request or reason for getting a bonus video game trivia question"
            }
        },
        "required": ["value"],
        "additionalProperties": False
    }
}

third_tool_json = {
    "name": "third_tool",
    "description": "Use this tool to record the participant's quiz score so their progress can be tracked",
    "parameters": {
        "type": "object",
        "properties": {
            "value": {
                "type": "string",
                "description": "The participant's current quiz score"
            }
        },
        "required": ["value"],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": my_tool_json},
    {"type": "function", "function": api_tool_json},
    {"type": "function", "function": third_tool_json},
]

## 7. Update your system prompt for three tools

**TODO:** Build your `system_prompt` f-string the same way you did in Module 5, still using
`persona` and `notes`, but now with three rules telling the model when to reach for each tool,
the same way Lab 5's system prompt added two new sentences to Lab 4's.

In [ ]:
system_prompt = f"""
# Your role

You are {persona}

Your function is to quiz a internet user about placement of video game genre popularity based on how much games it has

# Context

These are the genres of video game by rank you have to work:

{notes}

# Rules

Ask exactly ONE question at a time, based only on the notes above. Wait for the student's answer
before asking the next question.

After the student answers, clearly say whether they were right or wrong, and give a one-sentence
explanation either way, staying in character the whole time.

If the student's answer was wrong, or they say "I don't know", you must record the missed question
using my_tool before moving on to the next question.

If the student asks for a bonus question or wants a break from the study material, use api_tool
to fetch a bonus video game trivia question. This bonus question does not need to come from the
notes.

If the student provides or asks to record their current score, use third_tool to record the
participant's score.

IMPORTANT: Only ask quiz questions based on the notes above. If the student asks about something
outside the notes, and it is not a bonus question or a request to record their score, explain that
the information is not covered in this study set.

Stay professional, ethical, and in character throughout the quiz.
"""

display(Markdown(system_prompt))

## 8. The routing problem: which tool did the model actually call?

With three tools, `chat()` can no longer assume. It has to check `tool_call.function.name` and
call the matching Python function.

**TODO:** Copy the `call_tool(name, arguments)` dispatcher pattern from Lab 5, Step 9, but route
across your own three tool names instead.

In [23]:
def call_tool(name, arguments):
    # TODO: if/elif/else on `name`, calling the matching tool function with its argument(s)
    # pulled out of `arguments`, and returning a short string in every branch, including an
    # "Unknown tool" fallback in the else branch
    if name == "my_tool":
        return my_tool(arguments.get("value"))
    elif name == "api_tool":
        return api_tool(arguments.get("value"))
    elif name == "third_tool":
        return third_tool(arguments.get("value"))
    else:
        return f"uknown tool: {name}"
    pass

## 9. Update `chat()` to route across all three tools

**TODO:** Copy the finished agent loop from your Module 5 assignment. The `while` loop and the
`for tool_call in message_obj.tool_calls` loop do not change at all, the only new line is a call
to `call_tool(...)` in place of your old single hardcoded tool call.

In [26]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    # TODO: while response.choices[0].finish_reason == "tool_calls":
    #     - message_obj = response.choices[0].message
    #     - messages.append(message_obj)
    #     - for tool_call in message_obj.tool_calls:
    #           - name = tool_call.function.name
    #           - arguments = json.loads(tool_call.function.arguments)
    #           - result = call_tool(name, arguments)
    #           - messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})
    #     - response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message
        messages.append(message_obj)

        for tool_call in message_obj.tool_calls:
            name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)

            result = call_tool(name, arguments)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })

        response = azure_client.chat.completions.create(
            model=DEPLOYMENT,
            messages=messages,
            tools=tools
        )

    return response.choices[0].message.content

## 10. Try your multi-tool Persona Agent

Launch the chat interface and try to trigger all three tools in one session: something that
should hit your original Module 5 tool, something that should hit your new API-backed tool, and
something that should hit your third tool. Watch the terminal or output to see which tool fires
each time. Interrupt the cell when you're done.

In [27]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\Users\estev\Assignment 6\.venv\Lib\site-packages\gradio\queueing.py", line 848, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "c:\Users\estev\Assignment 6\.venv\Lib\site-packages\gradio\route_utils.py", line 417, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "c:\Users\estev\Assignment 6\.venv\Lib\site-packages\gradio\blocks.py", line 2698, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "c:\Users\estev\Assignment 6\.venv\Lib\site-packages\gradio\blocks.py", line 1961, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\estev\Assignment 6\.venv\Lib\site-packages\gradio\

## Reflection questions

Answer these in a new Markdown cell below.

1. What are your three tools, and what does each one do? Why did you choose the second and third
   tools you designed?
   - my first tool are responsible to remember the questions that the user got wrong or did not know and then save in a missed_questions.txt, my second one based on a open trivia api source are responsible to ask a bonus question in case of errors or personally asked from the user and finally my third tool is responsible for remember the score of the quiz on score.txt for showing the user in case it want to ask about it. I choose based on what to complement quizzes to be more complexes and fun, also functionalities that are common to think when it comes to quizzes.

2. Compare this assignment to Module 5. What was exactly the same as going from one tool to more
   than one? What, if anything, surprised you?
   - The exactly was the persona, notes, system prompt and agent loop mostly copied from assignment 5 code like the assignment ask me to. Since it has more tools to work in assignment 6 I was more suprised on the answers and how it manages to change for second and third tool in case it ask, feels more complete and lively AI.

3. Walk through what `call_tool` actually does the first time your agent calls your new
   API-backed tool. What would break if you deleted the `elif` branch for that tool?
   - It would break the system since the elif statments exist to recognizes the other second and third tool to send their specific functions and continue the chat, so call_tool would just skip directly to else returning the message unknown tool.

4. Open Lab 5's system prompt and compare it to yours. Both add rules for two new tools, using
   only plain English sentences and no code changes to the agent loop. Why does that work, in
   terms of what a system prompt actually is?
   - It works because a system prompt gives the instruction for the AI about their name, role/function and how to behave, but the responsible to actually executing the ai tool is the python code, so even if I add rules and new tools it would only be necessary to describe what I want the tool to behave or be without needing to mess with the agent loop. 


## Wrap-up

Before submitting:

- Make sure your `.env` file is **not** included in what you submit, it contains your API key.
- Run the notebook top to bottom one more time to confirm it works cleanly.
- Answer all four reflection questions.
- Rename this file to include your name, for example `Module6Assignment_JaneDoe.ipynb`, and
  submit it.

You've now made the same jump Lab 5 made: one tool to several, with a real dispatcher and a loop
that already scaled to handle it.